# ***05 · CASO: para cada ángulo de 0° a 180°***

## ***Averiguar cuántas partículas entran en cada grado***

Para cada evento se tiene $x=\cos\theta$, que se convierte a ángulo con $\theta=\arccos(x)$ (en grados), y se cuenta cuántos eventos caen en cada intervalo de $1^\circ$:

$$k^\circ\le\theta<(k+1)^\circ,\qquad k=0,1,2,\dots,179$$

El último intervalo ($179^\circ$–$180^\circ$) incluye también el $180^\circ$ exacto. Son **180 intervalos** y la suma de todos debe dar $N=10\,000$.

El conteo (`particulas_por_grado`, en `src/fbasym/analysis.py`) se hace con un ciclo `for` sobre los 180 grados y aquí se verifica con `np.histogram`.

In [ ]:
import sys
sys.path.append("../src")          # permite importar el paquete fbasym sin instalarlo

import numpy as np
import matplotlib.pyplot as plt
from fbasym import *
from fbasym import plots

n_events = 10000
np.random.seed(42)

# mismos eventos y mismo orden de generación que en el notebook 02
A_sim, B_sim = 1.0, 0.0
A_ideal, B_ideal = 1.0, 0.5
A_NC, B_NC = coeficientes(M_Z)
ev_sim, _ = rejection_sampling(A_sim, B_sim, n_events)
ev_ideal, _ = rejection_sampling(A_ideal, B_ideal, n_events)
ev_nc, _ = rejection_sampling(A_NC, B_NC, n_events)

casos = [("Distribución simétrica (solo fotón)", ev_sim, A_sim, B_sim),
         ("Asimetría propuesta", ev_ideal, A_ideal, B_ideal),
         (r"Corrientes neutras, $\sqrt{s}=m_Z$", ev_nc, A_NC, B_NC)]

columnas = []
for nombre, ev, A, B in casos:
    cuentas, theta = particulas_por_grado(ev)
    cuentas_np, _ = np.histogram(theta, bins=np.arange(0, 181, 1))
    print(nombre)
    print(f"   total contado = {cuentas.sum()} de {len(ev)} eventos")
    print(f"   coincide con np.histogram: {np.array_equal(cuentas, cuentas_np)}")
    print(f"   intervalo más poblado: {np.argmax(cuentas)}° - {np.argmax(cuentas)+1}° ({cuentas.max()} partículas)")
    print(f"   intervalo menos poblado: {np.argmin(cuentas)}° - {np.argmin(cuentas)+1}° ({cuentas.min()} partículas)")
    columnas.append(cuentas)

### ***Gráficas: partículas por grado***

Cada barra corresponde a un grado y la curva roja es la predicción teórica (integrando $f$ entre $\cos k^\circ$ y $\cos(k+1)^\circ$).

Por el cambio de variable, $dx=\sin\theta\,d\theta$, se cumple $\dfrac{dN}{d\theta}=\dfrac{dN}{dx}\sin\theta$. La forma no coincide con la del histograma en $x$: cerca de $0^\circ$ y $180^\circ$ hay pocas partículas por grado aunque en $x$ haya muchas, ya que ahí un grado ocupa muy poco $\Delta x$.

In [ ]:
fig = plots.figura_conteo_casos([(n, A, B, len(ev)) for n, ev, A, B in casos], columnas)
fig.savefig("../figures/05_conteo_por_grado_casos.png", dpi=150)
plt.show()

series = [('simétrica (solo fotón)', 'b-', columnas[0]),
          ('asimetría propuesta', 'g--', columnas[1]),
          (r'corrientes neutras ($m_Z$)', 'r:', columnas[2])]
fig = plots.figura_conteo_comparacion(series)
fig.savefig("../figures/05_conteo_comparacion.png", dpi=150)
plt.show()

### ***Lista completa: los 180 intervalos (corrientes neutras en $\sqrt{s}=m_Z$)***

In [ ]:
cuentas_nc = columnas[2]
print("grado        partículas    |  grado        partículas    |  grado        partículas")
for k in range(60):
    linea = ""
    for j in (k, k + 60, k + 120):
        linea += f"{j:3d}° - {j+1:3d}°   {cuentas_nc[j]:5d}        |  "
    print(linea.rstrip(" |"))

### ***Dependencia del conteo con la energía***

Como $A_{FB}$ depende de $\sqrt{s}$, el reparto por grado también. El $\theta$ pequeño corresponde a *forward* y el cercano a $180^\circ$ a *backward*.

In [ ]:
series_E = []
for E, est in [(80, 'b-'), (M_Z, 'g--'), (105, 'r:')]:
    A, B = coeficientes(E)
    ev, _ = rejection_sampling(A, B, n_events)
    cuentas, _ = particulas_por_grado(ev)
    series_E.append((rf'$\sqrt{{s}}={E:.1f}$ GeV  ($A_{{FB}}$ = {afb_teorica(E):+.2f})', est, cuentas))
    print(f"raiz_s = {E:8.4f} GeV:  forward (0°-90°) = {np.sum(cuentas[:90])}   backward (90°-180°) = {np.sum(cuentas[90:])}")

fig = plots.figura_conteo_comparacion(series_E, 'Conteo por grado a distintas energías',
                                      'Conteo acumulado a distintas energías')
fig.savefig("../figures/05_conteo_por_energia.png", dpi=150)
plt.show()

### ***Exportación***

Tabla de 180 filas con el conteo de cada caso, guardada en `data/conteo_angular.csv`.

In [ ]:
datos = np.column_stack([np.arange(0, 180), np.arange(1, 181)] + columnas)
np.savetxt("../data/conteo_angular.csv", datos, delimiter=",", fmt="%d",
           header="grado_inicio,grado_fin,simetrica,asimetria_propuesta,corrientes_neutras", comments="")
print("guardado: data/conteo_angular.csv  (180 filas)")

## ***Conclusiones del conjunto***

- el método de aceptación-rechazo reproduce las distribuciones teóricas en los tres casos ($\chi^2/\text{ndof}\approx1$ y residuos dentro de $\pm2\sigma$)
- al subir la energía hacia la resonancia, la interferencia $\gamma$–$Z$ inclina la distribución primero hacia *backward* y luego hacia *forward*
- en el polo de la $Z$ la interferencia se anula y $A_{FB}\approx0.017$ queda determinada por el término $ZZ$
- el error de $A_{FB}$ decrece como $1/\sqrt{N}$; con $N=10^5$ se recupera $\sin^2\theta_W$ con un error de unos $0.002$
- el conteo por grado reparte las 10 000 partículas en 180 intervalos y sigue la predicción teórica

**Limitaciones:** cálculo a nivel árbol, sin correcciones radiativas, sin radiación de estado inicial, sin efectos del detector, y solo con la forma angular (no la sección eficaz absoluta).